# RepoEval-function · Qwen2.5-Coder-1.5B · C1–C4

Runs the adaptive-retrieval cascade on **RepoEval function-body completion** with **Qwen2.5-Coder-1.5B** (vLLM), using the committed Qwen estimator. Generation is **FIM** (unchanged pipeline). Results are committed and pushed back to a results branch.

**Runtime:** GPU (T4/L4/A100). Set `Runtime → Change runtime type → GPU` first.

**Flow:** clone → install → provision RepoEval → run C1–C4 → push results.

Flip `SMOKE` off to go from a 3-instance check to the full 455.

## 1. Configuration — edit these

In [ ]:
# ---- experiment knobs ----
SMOKE = True          # True: 3 instances (quick check). False: all 455.
SMOKE_LIMIT = 3

MODEL = 'Qwen/Qwen2.5-Coder-1.5B'
MODEL_FAMILY = 'qwen'
DATASET = 'repoeval_function'
MAX_TOKENS = 512       # function bodies are multi-line; 512 avoids truncation
T_RAG = 0.9
TOP_K = 10
ESTIMATOR = 'models/estimator_qwen25_1.5b.lgb'   # committed Qwen estimator
CONFIGS = ['C1_no_retrieve', 'C2_always_retrieve', 'C3_card', 'C4_cascade']

# ---- git push (results -> branch) ----
GIT_BRANCH = 'results/repoeval-qwen15'   # branch the results get pushed to
GIT_USER_NAME = 'colab-runner'
GIT_USER_EMAIL = 'colab@example.com'
# Paste a GitHub Personal Access Token (repo scope) when prompted below.
# It is read via getpass so it is not stored in the notebook.
BRANCH_FROM = 'main'

REPO_HTTPS = 'github.com/Luca-Ionescu/rag-static-analysis-experiment.git'
WORK_DIR = '/content/rag-static-analysis-experiment'
RESULTS_SUBDIR = f'results/qwen25_1.5b_{DATASET}'
print('SMOKE' if SMOKE else 'FULL', '| model', MODEL, '| dataset', DATASET,
      '| max_tokens', MAX_TOKENS)

## 2. GPU sanity check

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as e:
    print('No GPU visible — set Runtime -> Change runtime type -> GPU.', e)

## 3. Token & clone
Paste a GitHub PAT (repo scope). Used only to clone+push over HTTPS.

In [ ]:
import os, getpass, subprocess
GH_TOKEN = getpass.getpass('GitHub PAT (repo scope): ').strip()
auth_url = f'https://{GH_TOKEN}@{REPO_HTTPS}'
if os.path.isdir(WORK_DIR):
    subprocess.run(['rm', '-rf', WORK_DIR], check=True)
subprocess.run(['git', 'clone', auth_url, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run(['git', 'checkout', BRANCH_FROM], check=True)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('cloned at', os.getcwd())
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True))

## 4. Install dependencies
vLLM + the project requirements. Takes several minutes.

In [ ]:
import subprocess, sys
# vLLM (brings a compatible torch) + the static/retrieval/metrics stack.
pkgs = [
    'vllm==0.10.2',
    'transformers>=4.55.2,<5.0',
    'tree-sitter==0.23.2', 'tree-sitter-python==0.23.6',
    'rank-bm25', 'python-Levenshtein', 'lightgbm',
    'jsonlines', 'click', 'pyflakes', 'tqdm', 'scipy', 'scikit-learn',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
print('deps installed')

## 5. Provision RepoEval-function
Clones microsoft/CodeT (sparse) and unzips the function-level task JSONLs + repositories into `data/repoeval/` where `load_repoeval` expects them.

In [ ]:
import os, subprocess, zipfile, glob
os.chdir(WORK_DIR)
os.makedirs('data/repoeval/datasets', exist_ok=True)
os.makedirs('data/repoeval/repositories', exist_ok=True)

if not os.path.exists('/content/CodeT'):
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',
                    'https://github.com/microsoft/CodeT.git','/content/CodeT'], check=True)
    subprocess.run(['git','-C','/content/CodeT','sparse-checkout','set','RepoCoder'], check=True)

RC = '/content/CodeT/RepoCoder'
with zipfile.ZipFile(f'{RC}/datasets/datasets.zip') as z:
    z.extractall('data/repoeval/datasets')
with zipfile.ZipFile(f'{RC}/repositories/function_level.zip') as z:
    z.extractall('data/repoeval/repositories')

ds = glob.glob('data/repoeval/datasets/function_level_completion_2k*.jsonl')
print('function-level task file:', ds)
print('repos provisioned:', len(os.listdir('data/repoeval/repositories')))

## 6. Verify the loader is lossless
Confirms the RepoEval fix: `x_left + ground_truth + x_right == file` for every instance.

In [ ]:
import sys, os
os.chdir(WORK_DIR)
sys.path.insert(0, 'src')
from pathlib import Path
from adaptive_retrieval.eval.datasets import load_repoeval
n=bad=xr=0
for inst in load_repoeval(task='function'):
    n += 1
    full = (Path('data/repoeval/repositories')/Path(*inst.target_file.split('/'))).read_text(errors='replace')
    if inst.x_left + inst.ground_truth + inst.x_right != full: bad += 1
    if inst.x_right.strip(): xr += 1
print(f'instances={n}  reconstruction_failures={bad}  with_right_context={xr}')
assert bad == 0, 'loader not lossless!'
print('OK — loader lossless')

## 7. Run C1–C4
Each config writes a per-instance JSONL via the existing `scripts/04_run_experiment.py`. C3/C4 use the Qwen estimator. A shared generation cache makes the zero-shot pass reusable across configs.

In [ ]:
import os, subprocess, sys
os.chdir(WORK_DIR)
os.makedirs(RESULTS_SUBDIR, exist_ok=True)
os.makedirs('data/generation_cache', exist_ok=True)

def run_config(cfg):
    out = f'{RESULTS_SUBDIR}/{cfg}.jsonl'
    cmd = [sys.executable, 'scripts/04_run_experiment.py',
           '--config', cfg, '--dataset', DATASET,
           '--backend', 'vllm', '--model', MODEL,
           '--model-family', MODEL_FAMILY,
           '--max-tokens', str(MAX_TOKENS),
           '--t-rag', str(T_RAG), '--top-k', str(TOP_K),
           '--output', out, '--cache-dir', 'data/generation_cache']
    if cfg in ('C3_card', 'C4_cascade'):
        cmd += ['--estimator-path', ESTIMATOR]
    if SMOKE:
        cmd += ['--limit', str(SMOKE_LIMIT)]
    print('>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

for cfg in CONFIGS:
    run_config(cfg)
print('all configs done')

## 8. Quick summary
Aggregates each config's JSONL (EM/ES/IdF1/hallucination/retrieval%).

In [ ]:
import os, json, sys
os.chdir(WORK_DIR)
sys.path.insert(0, 'src')
from adaptive_retrieval.eval.runner import aggregate_from_jsonl
rows = []
for cfg in CONFIGS:
    path = f'{RESULTS_SUBDIR}/{cfg}.jsonl'
    if not os.path.exists(path):
        continue
    s = aggregate_from_jsonl(path)
    rows.append((cfg, s.n_instances, round(s.percent_retrieval,1), s.metrics))
summary = {
    'model': MODEL, 'dataset': DATASET, 'smoke': SMOKE,
    'max_tokens': MAX_TOKENS, 't_rag': T_RAG,
    'configs': {r[0]: {'n': r[1], 'retrieval_pct': r[2], **r[3]} for r in rows},
}
with open(f'{RESULTS_SUBDIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 9. Commit & push results
`results/` is gitignored, so we **force-add** the run's JSONLs + summary and push to the results branch. The SMOKE run is tagged in the commit message.

In [ ]:
import os, subprocess
os.chdir(WORK_DIR)
subprocess.run(['git','config','user.name', GIT_USER_NAME], check=True)
subprocess.run(['git','config','user.email', GIT_USER_EMAIL], check=True)
# fresh results branch off main
subprocess.run(['git','checkout','-B', GIT_BRANCH], check=True)
# force-add: results/ is gitignored by design
subprocess.run(['git','add','-f', RESULTS_SUBDIR], check=True)
tag = 'SMOKE' if SMOKE else 'FULL'
msg = f'RepoEval-function Qwen-1.5B results ({tag}, max_tokens={MAX_TOKENS})'
rc = subprocess.run(['git','commit','-m', msg])
if rc.returncode == 0:
    subprocess.run(['git','push','-u','origin', GIT_BRANCH, '--force'], check=True)
    print('pushed to', GIT_BRANCH)
else:
    print('nothing to commit (no new results?)')